# 01 — TrustedRisk Quickstart

**5-minute walkthrough**: from a fresh checkout to a calibrated DecisionCard.

This notebook assumes you have:
- Cloned the repo and activated the project virtualenv
- `data/coefficients.json` present (W1 calibration artifact)
- `src` on `PYTHONPATH`

We'll exercise three core MCP tools end-to-end:

1. `compute_readmission_risk` — calibrated 30-day readmission probability + CI95 + valid_until
2. `compute_decision_utility` — translates the risk into a recommended action under explicit cost/utility assumptions
3. `detect_phi` — scrubs free-text input for PHI before any LLM polish

Every output is a Pydantic model with stable schemas under `src/shared/schemas.py`.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

REPO_ROOT = Path('.').resolve().parent.parent  # docs/notebooks → repo root
sys.path.insert(0, str(REPO_ROOT / 'src'))
os.environ.setdefault('TRUSTEDRISK_COEFFICIENTS_PATH',
                      str(REPO_ROOT / 'data' / 'coefficients.json'))

## 1. Calibrated readmission risk

LACE = Length of stay + Acuity + Comorbidity + ED visits. The tool returns a **bin-level** posterior derived from the W1 5-bin Beta-Binomial — same artifact that ships ECE 0.0078 on the calibration cohort.

In [ ]:
from mcp_server.tools.readmission_risk import compute_readmission_risk

risk = asyncio.run(compute_readmission_risk(
    lace_components={'L': 4, 'A': 3, 'C': 3, 'E': 1},
    patient_age=72, patient_sex='female',
))
print(f"30-day readmission risk: {risk.point_estimate_probability:.3f}")
print(f"CI95: [{risk.ci95_lower:.3f}, {risk.ci95_upper:.3f}]")
print(f"Valid until: {risk.valid_until}")

## 2. Decision utility

Maps the calibrated risk to one of `{discharge_home, home_with_care, snf, continued_admission}` under explicit per-action cost / utility assumptions. The tool surfaces the dominant action AND the utility delta vs the runner-up — useful when the boundary is fragile.

In [ ]:
from mcp_server.tools.decision_utility import compute_decision_utility

decision = asyncio.run(compute_decision_utility(
    risk_estimate=risk,
))
print(f"Recommended action: {decision.recommended_action}")
print(f"Utility delta vs runner-up: {decision.utility_delta_runner_up:.3f}")
print(f"Abstain recommended: {decision.abstain_recommended}")

## 3. PHI scrubbing

`detect_phi` is the cheapest stateless safety check we ship. It returns:
- a **redaction_map** (`'John Doe' → '[NAME]'`)
- per-entity offsets (so callers can highlight in a UI)
- a `risk_level` label aggregated across high/medium-risk types

It works without Presidio (regex fallback) and is the target for the v2 red-team campaign in notebook 03.

In [ ]:
from mcp_server.tools.detect_phi import detect_phi

raw = (
    "Patient John Smith MRN 0001234 DOB 1955-03-04 phone 212-555-0100 "
    "presented with chest pain radiating to the left arm."
)
phi = asyncio.run(detect_phi(text=raw))
print(f"Risk level: {phi.risk_level}")
print("Redaction map:")
for original, replacement in phi.redaction_map.items():
    print(f"  {original!r} → {replacement}")

## What's next

- Notebook **02** — full HEDIS / CMS Stars workflow (Phase 10.1).
- Notebook **03** — running the 110-prompt v2 red-team corpus (Phase 10.4).
- See `docs/WHITE_PAPER.md` for the system architecture overview.